In [2]:
import pandas as pd

# the processed data already has lyrics split into verbs, nouns, adverbs, corpus, words counts, unique word counts etc
processedData = pd.read_csv("../data/all_songs_data_processed.csv")

regData = pd.read_csv("../data/all_songs_data.csv")
print(regData.head(10))

                                               Album  \
0                              Battle of New Orleans   
1                                         That’s All   
2                     “Mr Personality’s” 15 Big Hits   
3                The Greatest Hits Of Frankie Avalon   
4                         Paul Anka Sings His Big 15   
5                              The Bobby Darin Story   
6                         Sweet Sounds By The Browns   
7  The Fleetwoods: Greatest Hits: Priceless Colle...   
8                                Let’s Work Together   
9  The Fleetwoods: Greatest Hits: Priceless Colle...   

                                           Album URL            Artist  \
0  https://genius.com/albums/Johnny-horton/Battle...     Johnny Horton   
1   https://genius.com/albums/Bobby-darin/That-s-all       Bobby Darin   
2  https://genius.com/albums/Lloyd-price/Mr-perso...       Lloyd Price   
3  https://genius.com/albums/Frankie-avalon/The-g...    Frankie Avalon   
4  https://ge

In [1]:
import pandas as pd

# the processed data already has lyrics split into verbs, nouns, adverbs, corpus, words counts, unique word counts etc
processedData = pd.read_csv("../data/all_songs_data_processed.csv")

regData = pd.read_csv("../data/all_songs_data.csv")
print(regData.head())

#clean dataset
#drop uneeded columns
regData = regData.drop(columns=['Media'])
regData = regData.drop(columns=['Song URL'])
regData = regData.drop(columns=['Album URL'])
regData = regData.drop(columns=['Writers'])
regData = regData.drop(columns=['Featured Artists'])
regData.head()

print(regData.dtypes)
#change Year and Release Date columns to correct types
#changing year to int
regData["Year"] = regData["Year"].astype("int64")
#change release date to date
regData["Release Date"] = pd.to_datetime(regData["Release Date"], errors='coerce')
regData["Release Date"] = regData["Release Date"].dt.normalize()

#see how many nulls are in each column
regData.isnull().sum()
#album- 464
#lyrics- 116
#release date- 1937
#6500 total rows

nolyric = regData[regData["Lyrics"].isnull()]
nolyric['Rank'].value_counts(sort=False)

#all of the rows that have no lyrics also have no album or release date, going to drop those becuase we need the lyrics to do this project
regData[regData["Lyrics"].isnull()]
regData[regData["Lyrics"].isna()]["Rank"].value_counts().sort_index()
regData[regData["Lyrics"].isna()]["Year"].value_counts().sort_index()
#lyric nulls seem to be fairly random so not to worried about there being any sort of correlation, and we need the lyrics to do the assingment so have to drop these ones
#drop null lyrics
regData = regData.dropna(subset=['Lyrics'])

regData.isnull().sum()
#now have 348 null albums,1821 null release dates
#look at album nulls
regData[regData["Album"].isnull()]
regData[regData["Album"].isna()]["Rank"].value_counts().sort_index()
regData[regData["Album"].isna()]["Year"].value_counts().sort_index()
#1960s have the most null albums, but going to leave these because Album doesn't really affect the analysis that much, I think its okay for there to be some nulls
#look at release date nulls
regData[regData["Release Date"].isnull()]
regData[regData["Release Date"].isna()]["Year"].value_counts().sort_index()
#theres a lot of null release dates, want to see if any are ranked in multiple years
nullDates = regData[regData['Release Date'].isna()]
nullDates['Song Title'].value_counts().head(60)
nullCounts = nullDates['Song Title'].value_counts()
nullCounts1 = nullCounts[nullCounts > 1]
nullDateRepeats = regData['Song Title'].isin(nullCounts1)
regData[nullDateRepeats]

#noticed one song had incorrect lyrics, going to fix that
regData['Song Title'].value_counts()
regData[regData["Song Title"] == "Stay"]
correctlyrics = regData.loc[
    (regData["Year"] == 2022) &
    (regData["Artist"] == "The Kid Laroi and Justin Bieber") &
    (regData["Album"] == "Fuser Soundtrack"),
    "Lyrics"
].iloc[0]
regData.loc[
    (regData["Year"] == 2021) &
    (regData["Artist"] == "The Kid Laroi and Justin Bieber") &
    (regData["Album"] == "NOW That’s What I Call Music! 96 [UK]"),
    "Lyrics"
] = correctlyrics







                                 Album  \
0                Battle of New Orleans   
1                           That’s All   
2       “Mr Personality’s” 15 Big Hits   
3  The Greatest Hits Of Frankie Avalon   
4           Paul Anka Sings His Big 15   

                                           Album URL          Artist  \
0  https://genius.com/albums/Johnny-horton/Battle...   Johnny Horton   
1   https://genius.com/albums/Bobby-darin/That-s-all     Bobby Darin   
2  https://genius.com/albums/Lloyd-price/Mr-perso...     Lloyd Price   
3  https://genius.com/albums/Frankie-avalon/The-g...  Frankie Avalon   
4  https://genius.com/albums/Paul-anka/Paul-anka-...       Paul Anka   

  Featured Artists                                             Lyrics  \
0               []  [Verse 1] In 1814 we took a little trip Along ...   
1               []  Oh the shark, babe Has such teeth, dear And he...   
2               []  Over and over I tried to prove my love to you ...   
3               []  He

In [ ]:
# splitting up the words
import spacy
nlp = spacy.load("en_core_web_sm")

#going to add a new column for the cleaned lyrics
regData['cleanedLyrics'] = (
    regData['Lyrics']
    .str.lower()    #puts them all in lowercase
    .str.replace(r"\[.*?\]", "", regex=True)     #removes everythign inside bracketes [ ]
    .str.replace(r"(chorus|verse|bridge|intro|outro|featuring)\s*\d*:", "", regex=True)     #remove any other 'chores', 'verse', etc. from the lyrics
    .str.replace(r"[^a-z\s]", "", regex=True)      #strips so only letters a-z and spaces are included
    .str.replace(r"\s+", " ", regex=True)          #remove extra spaces
    .str.strip()                                   #remove leading/trailing spaces
)

#make each word a token
regData['tokens'] = regData['cleanedLyrics'].apply(
    lambda text: [t.lemma_ for t in nlp(text) if t.is_alpha and not t.is_stop and len(t.lemma_) > 1]
    #the t.lemms_ converts words into their base form
    #the not t.is_stop removes 'stop words like and, is, you etc.
)



In [ ]:
#putting all the lyrics into english

from langdetect import detect

def safe_detect(text):
    try:
        return detect(text)
    except:
        return None

regData["language"] = regData["Lyrics"].apply(
    lambda x: safe_detect(x) if pd.notna(x) else None
)
regData["language"].value_counts()

#looking at all the different languages
regData[regData["language"] == "ro"]
#these are all instramentals- no lyrics so going to keep them
regData[regData["language"] == "ro"]["Rank"].value_counts().sort_index()
regData[regData["language"] == "ro"]["Year"].value_counts().sort_index()
#pretty randomly distributed, might have to remove all songs from before 1970s because they seem to have the most issues

#looking at spanish
regData[regData["language"] == "es"]
#a lot of these are not incorrect entries, just songs by spanish artists
# so going to keep them separate for now instead of translating them in place
spanishRows = regData[regData["language"] == "es"].copy()

#running into error with songs that have books instead of lyrics,
#maybe focus on correcting those, then come back to languages
#for the main analysis, keep only English songs for now
regData = regData[regData["language"] == "en"].copy()
regData.reset_index(drop=True, inplace=True)

In [ ]:
import matplotlib.pyplot as plt

#looking at the amount of tokens in each song
regData["tokenCount"] = regData["tokens"].apply(len)
regData["tokenCount"].describe()
regData.nlargest(100, "tokenCount")[["Song Title", "Year", "Artist", "tokenCount", "Lyrics"]]

#making a histogram of lyrics distrabution to better understand what we're dealing with
counts, bins, _ = plt.hist(regData["tokenCount"], bins=80)
counts = counts.astype(int)
print("num songs in each bin", counts)  # number of songs in each bin
print("bin edges", bins)    # bin edges
plt.show()

#some songs are extremely long, which may mean they are book-like entries or bad scrape data
#going to remove the top 1% as outliers so they do not distort the analysis
upper_cutoff = regData["tokenCount"].quantile(0.99)
print("Upper cutoff:", upper_cutoff)

#first make a new table to hold only those long songs
longsongs = regData[regData["tokenCount"] > upper_cutoff].copy()

#keep the cleaned dataset without those extreme outliers
regData = regData[regData["tokenCount"] <= upper_cutoff].copy()

#also remove very short entries that do not give enough text for analysis
regData = regData[regData["tokenCount"] >= 20].copy()

regData.reset_index(drop=True, inplace=True)
regData.head()



In [ ]:
#adding a few simple text features for the rest of the analysis
#these will help us compare songs by rank, year, and word usage

regData["uniqueWordCount"] = regData["tokens"].apply(lambda x: len(set(x)))
regData["avgWordLength"] = regData["tokens"].apply(
    lambda toks: sum(len(w) for w in toks) / len(toks) if len(toks) > 0 else 0
)
regData["lexicalRichness"] = regData.apply(
    lambda row: row["uniqueWordCount"] / row["tokenCount"] if row["tokenCount"] > 0 else 0,
    axis=1
)

regData[["Song Title", "Year", "Rank", "tokenCount", "uniqueWordCount", "avgWordLength", "lexicalRichness"]].head()

In [ ]:
#saving the cleaned dataset so everyone in the group can use the same version
#this should be the main file used for the rest of the project

regData.to_csv(
    "all_songs_data_cleaned.csv",
    index=False
)

print("cleaned dataset saved")

In [ ]:
#looking at the most common words in the cleaned lyrics
#this helps us see whether high-ranking songs tend to use different language

from collections import Counter

allTokens = [token for tokens in regData["tokens"] for token in tokens]
wordCounts = Counter(allTokens)

print(wordCounts.most_common(25))

#plotting the top 15 most common words
topWords = wordCounts.most_common(15)
words = [w for w, c in topWords]
counts = [c for w, c in topWords]

plt.figure(figsize=(10, 5))
plt.bar(words, counts)
plt.xticks(rotation=45, ha="right")
plt.title("Top 15 Most Common Words in Cleaned Lyrics")
plt.tight_layout()
plt.show()

Ways to analyze:
*   TF-IDF : identifies important words vs commonly used words, can find words that make a song unique
*   Sentiment analysis : determine 'happy' vs 'sad' songs
*   N-grams : analyzes sequences of words instead of the individual words
*   Topic modeling : automatically finds themes in the data
    * common method- LDA (Latent Dirichlet Allocation)
* word embeddings/semantic analysis - Word2vec or GloVe- analyze relationships between words based on conext- can find words used similarly across songs, anaylze lyrical style and semantics
* Clustering : can group songs into groups- love songs, party songs etc.
* Classification : can predict song's genre, artist, rank?
  * clustering and classification use machine learning after lyrics are converted to numerical features(TF-IDF)

